# Características RoCoLe — GLCM e índices RGB

Objetivo: medir textura (GLCM) y color (índices RGB) sobre las fotos ya partidas en train/val/test, primero en **una** hoja de train como comprobación, y después en todo el inventario.

Por ahora usamos la **fotografía completa** (hoja + fondo). Más adelante contrastaremos con una máscara de hoja. No entrenamos modelos aquí: eso va en los cuadernos 03 y 04.

Responsables: Víctor y Gio.


## Llegar al índice y fijar la semilla

Usamos la ruta al índice featured (ya tiene `split` y `photo_path`) y una semilla fija para cualquier muestreo.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from pycocotools.coco import COCO
from skimage.feature import graycomatrix, graycoprops
from tqdm.auto import tqdm


In [ ]:
repo_root = Path("../..").resolve()
featured_dir = repo_root / "data" / "featured" / "rocole"
index_path = featured_dir / "index.csv"
seed = 42

print("repo_root:", repo_root)
print("index_path:", index_path)
print("existe:", index_path.exists())
seed


## Cargar el índice partido

El índice es la mesa de trabajo: una fila por foto, con etiqueta, partición y ruta a la imagen curada. No rehacemos el split.


In [ ]:
rocole_index = pd.read_csv(index_path)

print("filas:", len(rocole_index))
print("columnas:", list(rocole_index.columns))
rocole_index.head()


## Elegir una sola foto de train

Para verificar el camino basta con una hoja de aprender. No miramos val ni test todavía.


In [ ]:
# rocole_index: mesa con File, split, photo_path y etiquetas

train_rows = rocole_index.loc[rocole_index["split"] == "train"].reset_index(drop=True)
sample_row = train_rows.iloc[0]

print("File:", sample_row["File"])
print("Binary.Label:", sample_row["Binary.Label"])
print("Multiclass.Label:", sample_row["Multiclass.Label"])
print("photo_path:", sample_row["photo_path"])
sample_row


## Abrir la fotografía completa

Leemos la imagen tal cual está en curated: hoja y fondo juntos. Si carga y se ve, el resto del cuaderno puede apoyarse en esa misma ruta.


In [ ]:
photo_path = repo_root / sample_row["photo_path"]
sample_image = Image.open(photo_path).convert("RGB")

print("ruta:", photo_path)
print("existe:", photo_path.exists())
print("tamaño:", sample_image.size, "modo:", sample_image.mode)
plt.figure(figsize=(6, 6))
plt.imshow(sample_image)
plt.axis("off")
plt.title(f"{sample_row['File']} · {sample_row['Binary.Label']}")
plt.show()
sample_image


## Pasar la foto a números

La imagen PIL no sirve para restas canal a canal. La convertimos a un arreglo float (0–255) para calcular los índices sin perder decimales.


In [ ]:
# float antes de restas: evita truncar en los índices
sample_rgb = np.asarray(sample_image, dtype=np.float64)

print("forma:", sample_rgb.shape, "dtype:", sample_rgb.dtype)
print("min/max:", sample_rgb.min(), sample_rgb.max())
sample_rgb


## Separar rojo, verde y azul

Cada índice mira cómo se relacionan los tres canales. Sacamos red_channel, green_channel y blue_channel del arreglo float.


In [ ]:
# sample_rgb: HxWx3 float

red_channel = sample_rgb[:, :, 0]
green_channel = sample_rgb[:, :, 1]
blue_channel = sample_rgb[:, :, 2]

print("red:", red_channel.shape, "green:", green_channel.shape, "blue:", blue_channel.shape)
red_channel, green_channel, blue_channel


## ExG: exceso de verde

Normalizamos cada canal por la suma RGB (con `eps` para no dividir por cero) y calculamos ExG = 2g − r − b. Aún sin máscara: hoja y fondo juntos.


In [ ]:
eps = 1e-8

channel_sum = red_channel + green_channel + blue_channel + eps
r = red_channel / channel_sum
g = green_channel / channel_sum
b = blue_channel / channel_sum
exg_map = 2.0 * g - r - b

print("exg_map:", exg_map.shape, "min/max:", exg_map.min(), exg_map.max())
exg_map


## NGRDI: verde contra rojo

NGRDI = (G − R) / (G + R + eps). Resalta dónde el verde gana al rojo, píxel a píxel.


In [ ]:
# green_channel, red_channel, eps

ngrdi_map = (green_channel - red_channel) / (green_channel + red_channel + eps)

print("ngrdi_map:", ngrdi_map.shape, "min/max:", ngrdi_map.min(), ngrdi_map.max())
ngrdi_map


## VARI: verde, rojo y azul

VARI = (G − R) / (G + R − B + eps). Parecido al NGRDI, pero resta el azul en el denominador.


In [ ]:
# green_channel, red_channel, blue_channel, eps

vari_map = (green_channel - red_channel) / (
    green_channel + red_channel - blue_channel + eps
)

print("vari_map:", vari_map.shape, "min/max:", vari_map.min(), vari_map.max())
vari_map


## Mirar el mapa ExG

Una sola figura del exceso de verde sobre la foto de ejemplo, para ver si el índice se comporta como esperamos antes de resumir números.


In [ ]:
# exg_map: mapa ExG de la foto de ejemplo
results_dir = repo_root / "experiments" / "results"
results_dir.mkdir(parents=True, exist_ok=True)
exg_fig_path = results_dir / "rocole_sample_exg_map.png"

fig, ax = plt.subplots(figsize=(6, 6))
im = ax.imshow(exg_map, cmap="RdYlGn")
ax.set_title(f"ExG · {sample_row['File']}")
ax.axis("off")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
fig.savefig(exg_fig_path, dpi=120, bbox_inches="tight")
plt.show()

print("figura:", exg_fig_path)
exg_fig_path


## Resumir los tres índices

Media, desviación, mínimo y máximo de ExG, NGRDI y VARI sobre toda la foto (sin máscara). Chequeo numérico antes de GLCM e inventario.


In [ ]:
# exg_map, ngrdi_map, vari_map

index_summary = {
    "exg": {
        "mean": float(exg_map.mean()),
        "std": float(exg_map.std()),
        "min": float(exg_map.min()),
        "max": float(exg_map.max()),
    },
    "ngrdi": {
        "mean": float(ngrdi_map.mean()),
        "std": float(ngrdi_map.std()),
        "min": float(ngrdi_map.min()),
        "max": float(ngrdi_map.max()),
    },
    "vari": {
        "mean": float(vari_map.mean()),
        "std": float(vari_map.std()),
        "min": float(vari_map.min()),
        "max": float(vari_map.max()),
    },
}

for name, stats in index_summary.items():
    print(
        f"{name}: mean={stats['mean']:.4f} std={stats['std']:.4f} "
        f"min={stats['min']:.4f} max={stats['max']:.4f}"
    )
index_summary


## Escala de grises para textura

La matriz de coocurrencia mira cómo cambian los grises vecinos, no el color. Pasamos `sample_image` a una sola banda.


In [ ]:
# sample_image: foto PIL RGB de ejemplo

# una sola banda: la textura GLCM no usa color
gray = np.asarray(sample_image.convert("L"), dtype=np.uint8)

print("forma:", gray.shape, "dtype:", gray.dtype)
print("min/max:", int(gray.min()), int(gray.max()))
gray


## Bajar a 32 niveles de gris

Con 256 valores la matriz sería enorme. Cuantizamos a 32 niveles (`gray_level_count`) para que el GLCM sea manejable.


In [ ]:
# gray: imagen en escala de grises 0–255
gray_level_count = 32

# reescalamos a enteros 0 … gray_level_count-1
quantized_gray = (
    gray.astype(np.float64) / 255.0 * (gray_level_count - 1)
).astype(np.uint8)

print("niveles pedidos:", gray_level_count)
print("valores únicos:", len(np.unique(quantized_gray)))
print("min/max:", int(quantized_gray.min()), int(quantized_gray.max()))
quantized_gray


## Matriz de coocurrencia (una dirección)

Contamos qué tan a menudo un gris aparece al lado de otro: distancia 1, ángulo 0 (horizontal). Una sola dirección como comprobación.


In [ ]:
# quantized_gray, gray_level_count

# distance=1, angle=0 rad (horizontal)
glcm = graycomatrix(
    quantized_gray,
    distances=[1],
    angles=[0],
    levels=gray_level_count,
)

print("forma GLCM:", glcm.shape)
glcm


## Contraste GLCM

El contraste resume cuánto saltan los grises entre vecinos. Más contraste: más cambios bruscos en la textura.


In [ ]:
# glcm: matriz de coocurrencia

glcm_contrast = float(graycoprops(glcm, "contrast")[0, 0])

print("glcm_contrast:", glcm_contrast)
glcm_contrast


## Homogeneidad GLCM

La homogeneidad sube cuando los vecinos se parecen. Es la lectura inversa del contraste a grandes rasgos.


In [ ]:
# glcm: matriz de coocurrencia

glcm_homogeneity = float(graycoprops(glcm, "homogeneity")[0, 0])

print("glcm_homogeneity:", glcm_homogeneity)
glcm_homogeneity


## Energía GLCM

La energía concentra el peso de la matriz: texturas más “ordenadas” tienden a valores más altos.


In [ ]:
# glcm: matriz de coocurrencia

glcm_energy = float(graycoprops(glcm, "energy")[0, 0])

print("glcm_energy:", glcm_energy)
glcm_energy


## Qué miden contraste, homogeneidad y energía

- **Contraste:** qué tan distintos son los grises vecinos; sube con bordes y manchas abruptas.
- **Homogeneidad:** qué tan parecidos son esos vecinos; sube en zonas suaves.
- **Energía:** qué tan concentrada está la coocurrencia; sube cuando pocos pares de grises dominan.

Una sola foto de train verifica que el camino corre. No diagnostica la hoja ni elige umbrales.


## ¿Salieron números finitos?

Comprobamos que contraste, homogeneidad y energía no sean NaN ni infinitos.


In [ ]:
# glcm_contrast, glcm_homogeneity, glcm_energy

glcm_props_ok = bool(
    np.isfinite(glcm_contrast)
    and np.isfinite(glcm_homogeneity)
    and np.isfinite(glcm_energy)
)

print("glcm_props_ok:", glcm_props_ok)
print(
    "contrast=", glcm_contrast,
    "homogeneity=", glcm_homogeneity,
    "energy=", glcm_energy,
)
glcm_props_ok


## Máscara oficial COCO (solo la hoja)

Hasta ahora medimos color y textura sobre la foto entera (hoja + fondo). Ahora contrastamos con la **máscara oficial COCO** de RoCoLe: sin rembg, sin umbral ExG, sin GrabCut. Seguimos con la misma foto de train.


## Cargar las anotaciones COCO

Usamos el JSON de segmentación del dataset para saber qué píxeles son hoja.


In [ ]:
# si falta: pip install pycocotools

coco_path = repo_root / "data/raw/RoCoLe/Annotations/RoCoLE-coco.json"
coco = COCO(str(coco_path))

print("coco_path:", coco_path)
print("existe:", coco_path.exists())
print("imágenes en COCO:", len(coco.getImgIds()))
coco


## Máscara de la hoja de ejemplo

Con el nombre de archivo de la foto de train pedimos a COCO la silueta oficial de esa hoja.


In [ ]:
# coco, sample_row, sample_image

target_file = sample_row["File"]
# En RoCoLe file_name es URL; emparejamos si termina en File (equivalente a file_name ↔ File)
matched_images = [
    img
    for img in coco.loadImgs(coco.getImgIds())
    if img["file_name"].endswith(target_file)
]
assert len(matched_images) == 1, (
    f"esperaba 1 imagen COCO para {target_file}, hay {len(matched_images)}"
)
coco_image = matched_images[0]
# categoría Leaf = 1
# imgIds en lista: los id de RoCoLe son strings (Labelbox)
ann_ids = coco.getAnnIds(imgIds=[coco_image["id"]], catIds=[1])
annotations = coco.loadAnns(ann_ids)
assert len(annotations) >= 1, f"sin anotaciones Leaf para {target_file}"

# Si hay varias hojas: unimos con OR (alternativa: la de mayor área)
leaf_mask = None
for annotation in annotations:
    mask_i = coco.annToMask(annotation).astype(bool)
    leaf_mask = mask_i if leaf_mask is None else (leaf_mask | mask_i)

height, width = sample_image.size[1], sample_image.size[0]
assert leaf_mask.shape == (height, width), (
    f"máscara {leaf_mask.shape} vs imagen {(height, width)}"
)

print("File:", target_file)
print("coco image_id:", coco_image["id"])
print("anotaciones Leaf:", len(annotations))
print("píxeles hoja:", int(leaf_mask.sum()), "de", leaf_mask.size)
leaf_mask


## Ver la máscara sobre la foto

Comprobamos a ojo que la silueta cubre la hoja y no el fondo.


In [ ]:
# sample_image, leaf_mask, sample_row
results_dir = repo_root / "experiments" / "results"
results_dir.mkdir(parents=True, exist_ok=True)
mask_fig_path = results_dir / "rocole_sample_leaf_mask_overlay.png"

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(sample_image)
axes[0].set_title(f"Foto · {sample_row['File']}")
axes[0].axis("off")
axes[1].imshow(sample_image)
axes[1].imshow(leaf_mask, alpha=0.45, cmap="Greens")
axes[1].set_title("Máscara COCO (hoja)")
axes[1].axis("off")
fig.tight_layout()
fig.savefig(mask_fig_path, dpi=120, bbox_inches="tight")
plt.show()

print("figura:", mask_fig_path)
mask_fig_path


## Índices solo dentro de la hoja

Los mapas ExG, NGRDI y VARI ya están; ahora media y desviación solo donde la máscara marca hoja.


In [ ]:
# exg_map, ngrdi_map, vari_map, leaf_mask

masked_index_summary = {
    "exg": {
        "mean": float(exg_map[leaf_mask].mean()),
        "std": float(exg_map[leaf_mask].std()),
    },
    "ngrdi": {
        "mean": float(ngrdi_map[leaf_mask].mean()),
        "std": float(ngrdi_map[leaf_mask].std()),
    },
    "vari": {
        "mean": float(vari_map[leaf_mask].mean()),
        "std": float(vari_map[leaf_mask].std()),
    },
}

for name, stats in masked_index_summary.items():
    print(f"{name}: mean={stats['mean']:.4f} std={stats['std']:.4f}")
masked_index_summary


## Gris recortado a la hoja

Para el GLCM enmascarado recortamos al bounding box de la máscara y ponemos en 0 lo que no es hoja; así el fondo lejano no entra en la textura.


In [ ]:
# gray, leaf_mask

rows, cols = np.where(leaf_mask)
y0, y1 = int(rows.min()), int(rows.max()) + 1
x0, x1 = int(cols.min()), int(cols.max()) + 1
leaf_gray_crop = gray[y0:y1, x0:x1].copy()
crop_mask = leaf_mask[y0:y1, x0:x1]
# fuera de hoja dentro del bbox → 0 (constante)
leaf_gray_crop[~crop_mask] = 0

print("bbox:", (y0, y1, x0, x1))
print("forma crop:", leaf_gray_crop.shape)
print("min/max:", int(leaf_gray_crop.min()), int(leaf_gray_crop.max()))
leaf_gray_crop


## Cuantizar el recorte a 32 niveles

Igual que en la foto completa: bajamos a 32 niveles para que la matriz GLCM sea manejable.


In [ ]:
# leaf_gray_crop, gray_level_count

leaf_quantized_gray = (
    leaf_gray_crop.astype(np.float64) / 255.0 * (gray_level_count - 1)
).astype(np.uint8)

print("niveles:", gray_level_count)
print("valores únicos:", len(np.unique(leaf_quantized_gray)))
print("min/max:", int(leaf_quantized_gray.min()), int(leaf_quantized_gray.max()))
leaf_quantized_gray


## GLCM del recorte enmascarado

Repetimos la matriz de coocurrencia con distancia 1 y ángulo 0, ahora sobre el recorte de la hoja.


In [ ]:
# leaf_quantized_gray, gray_level_count

leaf_glcm = graycomatrix(
    leaf_quantized_gray,
    distances=[1],
    angles=[0],
    levels=gray_level_count,
)

print("forma leaf_glcm:", leaf_glcm.shape)
leaf_glcm


## Textura GLCM solo en la hoja

Los mismos tres números que ya tenemos para la foto completa: contraste, homogeneidad y energía del recorte.


In [ ]:
# leaf_glcm

masked_glcm_summary = {
    "contrast": float(graycoprops(leaf_glcm, "contrast")[0, 0]),
    "homogeneity": float(graycoprops(leaf_glcm, "homogeneity")[0, 0]),
    "energy": float(graycoprops(leaf_glcm, "energy")[0, 0]),
}

print(masked_glcm_summary)
masked_glcm_summary


## Completa vs máscara COCO

Ponemos lado a lado lo medido en la foto entera y lo medido solo en la hoja, para ver qué se mueve.


In [ ]:
# index_summary, masked_index_summary
# glcm_contrast, glcm_homogeneity, glcm_energy, masked_glcm_summary

feature_comparison = pd.DataFrame(
    [
        {
            "feature": "exg_mean",
            "full_image": index_summary["exg"]["mean"],
            "coco_mask": masked_index_summary["exg"]["mean"],
        },
        {
            "feature": "exg_std",
            "full_image": index_summary["exg"]["std"],
            "coco_mask": masked_index_summary["exg"]["std"],
        },
        {
            "feature": "ngrdi_mean",
            "full_image": index_summary["ngrdi"]["mean"],
            "coco_mask": masked_index_summary["ngrdi"]["mean"],
        },
        {
            "feature": "ngrdi_std",
            "full_image": index_summary["ngrdi"]["std"],
            "coco_mask": masked_index_summary["ngrdi"]["std"],
        },
        {
            "feature": "vari_mean",
            "full_image": index_summary["vari"]["mean"],
            "coco_mask": masked_index_summary["vari"]["mean"],
        },
        {
            "feature": "vari_std",
            "full_image": index_summary["vari"]["std"],
            "coco_mask": masked_index_summary["vari"]["std"],
        },
        {
            "feature": "glcm_contrast",
            "full_image": glcm_contrast,
            "coco_mask": masked_glcm_summary["contrast"],
        },
        {
            "feature": "glcm_homogeneity",
            "full_image": glcm_homogeneity,
            "coco_mask": masked_glcm_summary["homogeneity"],
        },
        {
            "feature": "glcm_energy",
            "full_image": glcm_energy,
            "coco_mask": masked_glcm_summary["energy"],
        },
    ]
)

print(feature_comparison.to_string(index=False))


## Conclusiones

En la misma foto de train, al medir solo la hoja con la máscara COCO los índices dejan de mezclarse con el fondo y la textura GLCM ya no “ve” pared o suelo; el borde hoja–constante del recorte puede empujar un poco el contraste. Es una sola hoja: el inventario y los modelos siguen después.


## Piloto: tres sanas y tres afectadas (train), dos protocolos

Antes del inventario completo, medimos color y textura en un puñado de hojas de **train**: tres sanas y tres afectadas. Las **mismas** seis fotos se miden dos veces: solo hoja (máscara COCO) y foto entera (sin segmentar). No es el dataset entero; es un chequeo de cuánto cambia el número al quitar el fondo.


## Tres hojas sanas de train

Sacamos tres fotos con `Binary.Label == healthy` solo del split de aprender, con la semilla fija del cuaderno.


In [ ]:
if "train_rows" not in globals():
    train_rows = rocole_index.loc[rocole_index["split"] == "train"].reset_index(drop=True)

healthy_train_sample = (
    train_rows.loc[train_rows["Binary.Label"] == "healthy"]
    .sample(n=3, random_state=seed)
    .reset_index(drop=True)
)

print("healthy_train_sample:", len(healthy_train_sample))
healthy_train_sample[["File", "Binary.Label", "split", "photo_path"]]


## Tres hojas afectadas de train

Igual que arriba, pero con `Binary.Label == unhealthy`: tres fotos de aprender afectadas.


In [ ]:
# train_rows, seed

unhealthy_train_sample = (
    train_rows.loc[train_rows["Binary.Label"] == "unhealthy"]
    .sample(n=3, random_state=seed)
    .reset_index(drop=True)
)

print("unhealthy_train_sample:", len(unhealthy_train_sample))
unhealthy_train_sample[["File", "Binary.Label", "split", "photo_path"]]


## Unir el puñado piloto

Juntamos las tres sanas y las tres afectadas en una sola mesa de seis filas: `pilot_train_rows`.


In [ ]:
# healthy_train_sample, unhealthy_train_sample

pilot_train_rows = pd.concat(
    [healthy_train_sample, unhealthy_train_sample],
    ignore_index=True,
)

print("pilot_train_rows:", len(pilot_train_rows))
print(pilot_train_rows["Binary.Label"].value_counts().to_string())
pilot_train_rows[["File", "Binary.Label", "split"]]


## Funciones de resumen RGB y GLCM (con o sin máscara)

Si pasamos máscara, medimos solo la hoja (índices en píxeles de hoja; GLCM en bbox con fuera-de-hoja en 0). Si `leaf_mask` es `None`, el promedio y el GLCM usan **toda** la foto, igual que en la exploración previa del cuaderno.


In [ ]:
# eps, gray_level_count; graycomatrix / graycoprops ya importados arriba

def extract_rgb_summary(image_rgb_or_pil, leaf_mask=None):
    """Media y std de ExG, NGRDI y VARI; leaf_mask=None → foto completa."""
    if hasattr(image_rgb_or_pil, "convert"):
        image_rgb = np.asarray(image_rgb_or_pil.convert("RGB"), dtype=np.float64)
    else:
        image_rgb = np.asarray(image_rgb_or_pil, dtype=np.float64)

    red_channel = image_rgb[:, :, 0]
    green_channel = image_rgb[:, :, 1]
    blue_channel = image_rgb[:, :, 2]

    channel_sum = red_channel + green_channel + blue_channel + eps
    r = red_channel / channel_sum
    g = green_channel / channel_sum
    b = blue_channel / channel_sum
    exg_map = 2.0 * g - r - b
    ngrdi_map = (green_channel - red_channel) / (green_channel + red_channel + eps)
    vari_map = (green_channel - red_channel) / (
        green_channel + red_channel - blue_channel + eps
    )

    # sin máscara: promediamos toda la foto
    if leaf_mask is None:
        return {
            "exg_mean": float(exg_map.mean()),
            "exg_std": float(exg_map.std()),
            "ngrdi_mean": float(ngrdi_map.mean()),
            "ngrdi_std": float(ngrdi_map.std()),
            "vari_mean": float(vari_map.mean()),
            "vari_std": float(vari_map.std()),
        }

    return {
        "exg_mean": float(exg_map[leaf_mask].mean()),
        "exg_std": float(exg_map[leaf_mask].std()),
        "ngrdi_mean": float(ngrdi_map[leaf_mask].mean()),
        "ngrdi_std": float(ngrdi_map[leaf_mask].std()),
        "vari_mean": float(vari_map[leaf_mask].mean()),
        "vari_std": float(vari_map[leaf_mask].std()),
    }


def extract_glcm_summary(gray_or_pil, leaf_mask=None):
    """GLCM d=1, ángulo 0, 32 niveles; leaf_mask=None → gris completa."""
    if hasattr(gray_or_pil, "convert"):
        gray = np.asarray(gray_or_pil.convert("L"), dtype=np.uint8)
    else:
        gray = np.asarray(gray_or_pil, dtype=np.uint8)

    if leaf_mask is None:
        # foto completa: sin bbox ni ceros artificiales
        work_gray = gray
    else:
        rows, cols = np.where(leaf_mask)
        y0, y1 = int(rows.min()), int(rows.max()) + 1
        x0, x1 = int(cols.min()), int(cols.max()) + 1
        work_gray = gray[y0:y1, x0:x1].copy()
        crop_mask = leaf_mask[y0:y1, x0:x1]
        work_gray[~crop_mask] = 0

    quantized_gray = (
        work_gray.astype(np.float64) / 255.0 * (gray_level_count - 1)
    ).astype(np.uint8)

    glcm = graycomatrix(
        quantized_gray,
        distances=[1],
        angles=[0],
        levels=gray_level_count,
    )
    return {
        "glcm_contrast": float(graycoprops(glcm, "contrast")[0, 0]),
        "glcm_homogeneity": float(graycoprops(glcm, "homogeneity")[0, 0]),
        "glcm_energy": float(graycoprops(glcm, "energy")[0, 0]),
    }


extract_rgb_summary, extract_glcm_summary


## Cargar la máscara COCO por File

En el JSON, `file_name` es una URL; emparejamos si termina en el `File` del índice (igual que en la sección COCO). Devolvemos la unión de anotaciones Leaf.


In [ ]:
# coco: anotaciones RoCoLe ya cargadas

def load_coco_leaf_mask(coco, file_name, height, width):
    """Máscara Leaf (OR) emparejando file_name del JSON por sufijo File."""
    matched_images = [
        img
        for img in coco.loadImgs(coco.getImgIds())
        if img["file_name"].endswith(file_name)
    ]
    if len(matched_images) != 1:
        raise ValueError(
            f"esperaba 1 imagen COCO para {file_name}, hay {len(matched_images)}"
        )

    coco_image = matched_images[0]
    # categoría Leaf = 1; imgIds en lista (ids Labelbox son strings)
    ann_ids = coco.getAnnIds(imgIds=[coco_image["id"]], catIds=[1])
    annotations = coco.loadAnns(ann_ids)
    if len(annotations) < 1:
        raise ValueError(f"sin anotaciones Leaf para {file_name}")

    leaf_mask = None
    for annotation in annotations:
        mask_i = coco.annToMask(annotation).astype(bool)
        leaf_mask = mask_i if leaf_mask is None else (leaf_mask | mask_i)

    if leaf_mask.shape != (height, width):
        raise ValueError(
            f"máscara {leaf_mask.shape} vs imagen {(height, width)} para {file_name}"
        )
    return leaf_mask


load_coco_leaf_mask


## Features con máscara COCO, fila a fila

Primero sacamos color y textura **solo con máscara COCO** en las seis filas de `pilot_train_rows`. Si falta máscara o falla la lectura, NaN en features y mensaje en `error_masked`.


In [ ]:
# pilot_train_rows, coco, repo_root
# extract_rgb_summary, extract_glcm_summary, load_coco_leaf_mask

feature_columns = [
    "exg_mean",
    "exg_std",
    "ngrdi_mean",
    "ngrdi_std",
    "vari_mean",
    "vari_std",
    "glcm_contrast",
    "glcm_homogeneity",
    "glcm_energy",
]

pilot_masked_feature_rows = []
for _, row in pilot_train_rows.iterrows():
    feature_row = {
        "File": row["File"],
        "Binary.Label": row["Binary.Label"],
        "split": row["split"],
        "photo_path": row["photo_path"],
        "error_masked": None,
    }
    for column_name in feature_columns:
        feature_row[column_name] = np.nan

    try:
        photo_path = repo_root / row["photo_path"]
        sample_image = Image.open(photo_path).convert("RGB")
        height, width = sample_image.size[1], sample_image.size[0]
        leaf_mask = load_coco_leaf_mask(coco, row["File"], height, width)
        rgb_summary = extract_rgb_summary(sample_image, leaf_mask)
        glcm_summary = extract_glcm_summary(sample_image, leaf_mask)
        feature_row.update(rgb_summary)
        feature_row.update(glcm_summary)
    except Exception as exc:
        # fallo de máscara o lectura: NaN en features, mensaje en error_masked
        feature_row["error_masked"] = str(exc)

    pilot_masked_feature_rows.append(feature_row)

print("pilot_masked_feature_rows:", len(pilot_masked_feature_rows))
print("con error_masked:", sum(1 for r in pilot_masked_feature_rows if r["error_masked"]))
pilot_masked_feature_rows


## Tabla del piloto con máscara

Pasamos la lista enmascarada a un DataFrame: `pilot_masked_feature_table`.


In [ ]:
# pilot_masked_feature_rows: lista de dicts del bucle con máscara

pilot_masked_feature_table = pd.DataFrame(pilot_masked_feature_rows)

print("filas:", len(pilot_masked_feature_table))
print("columnas:", list(pilot_masked_feature_table.columns))
pilot_masked_feature_table


## Features en foto completa, fila a fila

Ahora las **mismas** seis fotos, midiendo RGB+GLCM sobre la imagen entera (`leaf_mask=None`). Este protocolo no depende de COCO.


In [ ]:
# pilot_train_rows, repo_root, extract_rgb_summary, extract_glcm_summary
# feature_columns (misma lista que el protocolo enmascarado)

pilot_full_feature_rows = []
for _, row in pilot_train_rows.iterrows():
    feature_row = {
        "File": row["File"],
        "Binary.Label": row["Binary.Label"],
        "split": row["split"],
        "photo_path": row["photo_path"],
        "error_full": None,
    }
    for column_name in feature_columns:
        feature_row[column_name] = np.nan

    try:
        photo_path = repo_root / row["photo_path"]
        sample_image = Image.open(photo_path).convert("RGB")
        # sin máscara: misma API, leaf_mask=None
        rgb_summary = extract_rgb_summary(sample_image, leaf_mask=None)
        glcm_summary = extract_glcm_summary(sample_image, leaf_mask=None)
        feature_row.update(rgb_summary)
        feature_row.update(glcm_summary)
    except Exception as exc:
        # fallo de lectura: NaN en features, mensaje en error_full
        feature_row["error_full"] = str(exc)

    pilot_full_feature_rows.append(feature_row)

print("pilot_full_feature_rows:", len(pilot_full_feature_rows))
print("con error_full:", sum(1 for r in pilot_full_feature_rows if r["error_full"]))
pilot_full_feature_rows


## Tabla del piloto sin máscara

Pasamos la lista de foto completa a un DataFrame: `pilot_full_feature_table`.


In [ ]:
# pilot_full_feature_rows: lista de dicts del bucle sin máscara

pilot_full_feature_table = pd.DataFrame(pilot_full_feature_rows)

print("filas:", len(pilot_full_feature_table))
print("columnas:", list(pilot_full_feature_table.columns))
pilot_full_feature_table


## Comparar máscara vs foto completa

Juntamos las dos mesas por `File` para leer lado a lado: mismas features con sufijo `_masked` y `_full`.


In [ ]:
# pilot_masked_feature_table, pilot_full_feature_table, feature_columns

id_columns = ["File", "Binary.Label", "split", "photo_path"]
masked_part = pilot_masked_feature_table[
    id_columns + feature_columns + ["error_masked"]
].rename(columns={name: f"{name}_masked" for name in feature_columns})
full_part = pilot_full_feature_table[
    ["File"] + feature_columns + ["error_full"]
].rename(columns={name: f"{name}_full" for name in feature_columns})

pilot_compare_table = masked_part.merge(full_part, on="File", how="inner")

print("filas:", len(pilot_compare_table))
print("columnas:", list(pilot_compare_table.columns))
pilot_compare_table


## Diferencias full − masked

Restamos foto completa menos máscara en cada feature numérica para ver el tamaño del cambio por foto.


In [ ]:
# pilot_compare_table, feature_columns

delta_rows = []
for _, row in pilot_compare_table.iterrows():
    delta_row = {
        "File": row["File"],
        "Binary.Label": row["Binary.Label"],
        "split": row["split"],
    }
    for column_name in feature_columns:
        delta_row[f"{column_name}_delta"] = (
            row[f"{column_name}_full"] - row[f"{column_name}_masked"]
        )
    delta_rows.append(delta_row)

pilot_feature_delta_table = pd.DataFrame(delta_rows)

print("filas:", len(pilot_feature_delta_table))
print("columnas:", list(pilot_feature_delta_table.columns))
pilot_feature_delta_table


## Lectura del piloto (máscara vs completa)

En este puñado de seis hojas de train, mira `pilot_compare_table` y `pilot_feature_delta_table`: ¿los índices y el GLCM se mueven mucho al quitar la máscara? Es orientación de seis fotos de aprender, no una conclusión del dataset ni un ganador estadístico.


## Nota de decisión

El **experimento** puede reportar ambos protocolos (máscara COCO vs foto completa) para transparencia. El **modelo / pipeline de producción** principal se plantea **sin máscara** (sin segmentador). El inventario completo de features sigue después, fuera de este piloto.


## Inventario completo: RGB + GLCM sin máscara

Medimos color y textura en **todas** las fotos del índice (`rocole_index`), con el protocolo de producción: foto entera, sin segmentar. Primero un smoke de pocas filas de train; luego el barrido completo y el CSV en `data/featured/rocole/`.


## Protocolo de producción (foto completa)

Fijamos en un diccionario lo que usará el inventario: sin máscara, 32 niveles de gris, GLCM a distancia 1 y ángulo 0, y el mismo `eps` que arriba.


In [ ]:
# eps, gray_level_count (ya fijados arriba en el cuaderno)

production_protocol = {
    "leaf_mask": None,
    "gray_level_count": 32,
    "distances": [1],
    "angles": [0],
    "eps": eps,
}

production_protocol


## Columnas de features del inventario

Lista fija de medidas que deben salir de las extractoras: media y std de los tres índices RGB, más contraste, homogeneidad y energía GLCM. Las funciones del piloto ya devuelven std; aquí solo nombramos las columnas.


In [ ]:
# extract_rgb_summary, extract_glcm_summary (piloto; ya incluyen mean y std)

feature_columns = [
    "exg_mean",
    "exg_std",
    "ngrdi_mean",
    "ngrdi_std",
    "vari_mean",
    "vari_std",
    "glcm_contrast",
    "glcm_homogeneity",
    "glcm_energy",
]

feature_columns


## Columnas de identidad que viajan con cada fila

Además de las features, cada fila del inventario lleva el archivo, el split, el grupo de enlace y las dos etiquetas del índice.


In [ ]:
id_columns = [
    "File",
    "split",
    "link_group",
    "Binary.Label",
    "Multiclass.Label",
]

id_columns


## Filas de smoke (train)

Para probar el camino sin recorrer las ~1481 fotos, reutilizamos `pilot_train_rows` si existe; si no, tomamos seis filas de train al azar con la semilla fija.


In [ ]:
# pilot_train_rows (si el piloto corrió), train_rows / rocole_index, seed

if "pilot_train_rows" in globals() and len(pilot_train_rows) > 0:
    smoke_train_rows = pilot_train_rows.copy()
else:
    # respaldo: seis de train con semilla fija
    if "train_rows" not in globals():
        train_rows = rocole_index.loc[rocole_index["split"] == "train"].reset_index(
            drop=True
        )
    smoke_train_rows = train_rows.sample(n=6, random_state=seed).reset_index(drop=True)

print("smoke_train_rows:", len(smoke_train_rows))
print(smoke_train_rows["Binary.Label"].value_counts().to_string())
smoke_train_rows[id_columns]


## Smoke: features sin máscara, fila a fila

Sobre esas pocas filas de train medimos RGB+GLCM con `leaf_mask=None` (protocolo de producción). Si falla una foto, NaN en features y mensaje en `error`.


In [ ]:
# smoke_train_rows, repo_root, production_protocol
# extract_rgb_summary, extract_glcm_summary, feature_columns, id_columns

smoke_feature_rows = []
for _, row in smoke_train_rows.iterrows():
    feature_row = {column_name: row[column_name] for column_name in id_columns}
    feature_row["error"] = None
    for column_name in feature_columns:
        feature_row[column_name] = np.nan

    try:
        photo_path = repo_root / row["photo_path"]
        sample_image = Image.open(photo_path).convert("RGB")
        # protocolo producción: sin máscara
        rgb_summary = extract_rgb_summary(
            sample_image, leaf_mask=production_protocol["leaf_mask"]
        )
        glcm_summary = extract_glcm_summary(
            sample_image, leaf_mask=production_protocol["leaf_mask"]
        )
        feature_row.update(rgb_summary)
        feature_row.update(glcm_summary)
    except Exception as exc:
        feature_row["error"] = str(exc)

    smoke_feature_rows.append(feature_row)

print("smoke_feature_rows:", len(smoke_feature_rows))
print("con error:", sum(1 for r in smoke_feature_rows if r["error"]))
smoke_feature_rows


## Tabla smoke y chequeo de columnas / finitos

Pasamos el smoke a DataFrame y comprobamos que estén todas las columnas pedidas y que, sin error, los números sean finitos.


In [ ]:
# smoke_feature_rows, feature_columns, id_columns

smoke_feature_table = pd.DataFrame(smoke_feature_rows)

expected_columns = id_columns + feature_columns + ["error"]
missing_columns = [
    name for name in expected_columns if name not in smoke_feature_table.columns
]
assert missing_columns == [], f"faltan columnas: {missing_columns}"

ok_mask = smoke_feature_table["error"].isna()
finite_ok = True
if ok_mask.any():
    feature_block = smoke_feature_table.loc[ok_mask, feature_columns]
    finite_ok = bool(np.isfinite(feature_block.to_numpy(dtype=float)).all())

print("filas:", len(smoke_feature_table))
print("columnas esperadas ok:", missing_columns == [])
print("finitos en filas sin error:", finite_ok)
smoke_feature_table


## Barrido completo del índice (sin máscara)

Recorremos **todas** las filas de `rocole_index` con el mismo protocolo de producción. Cada fallo queda en `error` y las features en NaN; no usamos máscaras ni miramos el test para elegir nada.


In [ ]:
# rocole_index, repo_root, production_protocol
# extract_rgb_summary, extract_glcm_summary, feature_columns, id_columns

try:
except ImportError:
    def tqdm(iterable, **kwargs):
        return iterable

inventory_feature_rows = []
for _, row in tqdm(
    rocole_index.iterrows(),
    total=len(rocole_index),
    desc="inventario RGB+GLCM",
):
    feature_row = {column_name: row[column_name] for column_name in id_columns}
    feature_row["error"] = None
    for column_name in feature_columns:
        feature_row[column_name] = np.nan

    try:
        photo_path = repo_root / row["photo_path"]
        sample_image = Image.open(photo_path).convert("RGB")
        rgb_summary = extract_rgb_summary(
            sample_image, leaf_mask=production_protocol["leaf_mask"]
        )
        glcm_summary = extract_glcm_summary(
            sample_image, leaf_mask=production_protocol["leaf_mask"]
        )
        feature_row.update(rgb_summary)
        feature_row.update(glcm_summary)
    except Exception as exc:
        feature_row["error"] = str(exc)

    inventory_feature_rows.append(feature_row)

print("inventory_feature_rows:", len(inventory_feature_rows))
print(
    "con error:",
    sum(1 for r in inventory_feature_rows if r["error"] is not None),
)


## Mesa de features del inventario

Convertimos la lista del barrido en un DataFrame: `rgb_glcm_features`.


In [ ]:
# inventory_feature_rows

rgb_glcm_features = pd.DataFrame(inventory_feature_rows)

print("filas:", len(rgb_glcm_features))
print("columnas:", list(rgb_glcm_features.columns))
rgb_glcm_features.head()


## Chequeo de filas del inventario

Comprobamos que haya una fila por foto del índice, que los conteos por split cuadren y cuántas filas trajeron error.


In [ ]:
# rgb_glcm_features, rocole_index

inventory_row_check = {
    "n_index": len(rocole_index),
    "n_features": len(rgb_glcm_features),
    "len_match": len(rgb_glcm_features) == len(rocole_index),
    "split_counts_index": rocole_index["split"].value_counts().to_dict(),
    "split_counts_features": rgb_glcm_features["split"].value_counts().to_dict(),
    "n_errors": int(rgb_glcm_features["error"].notna().sum()),
}

assert inventory_row_check["len_match"], (
    f"filas features {inventory_row_check['n_features']} "
    f"!= índice {inventory_row_check['n_index']}"
)

inventory_row_check


## Guardar el CSV de features

Escribimos `rgb_glcm_features.csv` en la carpeta featured de RoCoLe. No entrenamos nada aquí.


In [ ]:
# rgb_glcm_features, featured_dir

features_csv_path = featured_dir / "rgb_glcm_features.csv"
rgb_glcm_features.to_csv(features_csv_path, index=False)

print("guardado:", features_csv_path)
print("existe:", features_csv_path.exists())
features_csv_path


## Conclusiones

Quedó un CSV de RGB+GLCM **sin máscara** para las ~1481 fotos del índice (mismo protocolo de producción del piloto full); el smoke de train y el chequeo de filas/errores confirman el camino antes de cualquier modelo.
